In [27]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

--2026-10-17 18:11:46--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.1’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.04s   

2026-10-17 18:11:47 (6.42 MB/s) - ‘course_lead_scoring_2026.csv.1’ saved [278746/278746]



In [37]:
import pandas as pd
df=pd.read_csv('course_lead_scoring_2026.csv')
df.head(5)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [29]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [30]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [31]:
categorical_columns=['lead_source','industry','employment_status','location']
df[categorical_columns] = df[categorical_columns].fillna('NA')

numerical_columns=df.select_dtypes(include='number').columns
df[numerical_columns] = df[numerical_columns].fillna(0.0)
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [32]:
df['industry'].mode()

0    technology
Name: industry, dtype: str

In [33]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [34]:
numerical = df.select_dtypes(include='number')
corr = numerical.corr()

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for a, b in pairs:
    print(a, b, corr.loc[a, b])

interaction_count lead_score 0.9157457980418697
number_of_courses_viewed lead_score 0.7572042392104886
number_of_courses_viewed interaction_count 0.7216090038937035
annual_income interaction_count 0.12284235135955959


In [35]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)


In [36]:
y_train = df_train['converted'].reset_index(drop=True)
df_train = df_train.drop(columns=['converted']).reset_index(drop=True)

from sklearn.metrics import mutual_info_score

categorical = df_train.select_dtypes(include='str').columns

scores = {}
for col in categorical:
    scores[col] = round(mutual_info_score(y_train, df_train[col]), 2)

print(pd.Series(scores).sort_values(ascending=False))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


In [38]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = df.copy()

categorical_cols = df.select_dtypes(
    include=['object', 'str']
).columns.tolist()

numerical_cols = df.select_dtypes(
    include='number'
).columns.tolist()

categorical_cols = [
    col for col in categorical_cols if col != 'converted'
]
numerical_cols = [
    col for col in numerical_cols if col != 'converted'
]

df[categorical_cols] = df[categorical_cols].fillna('NA')
df[numerical_cols] = df[numerical_cols].fillna(0.0)

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train['converted']
y_val = df_val['converted']

df_train = df_train.drop(columns=['converted'])
df_val = df_val.drop(columns=['converted'])

In [42]:
dv = DictVectorizer()

X_train = dv.fit_transform(
    df_train.to_dict(orient='records')
)
X_val = dv.transform(
    df_val.to_dict(orient='records')
)

model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)
X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

X_val.indices = X_val.indices.astype(np.int32)
X_val.indptr = X_val.indptr.astype(np.int32)

model.fit(X_train, y_train)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print(round(accuracy_score(y_val, y_pred), 2))

0.65


In [41]:
print(y_train.isna().sum())
print(y_train.unique())
print(X_train.shape)
print(X_train.dtype if hasattr(X_train, "dtype") else X_train.shape)
import numpy as np

print(np.isnan(X_train.data).sum())
print(np.isinf(X_train.data).sum())
print(len(y_train), X_train.shape[0])

0
[1 0]
(3000, 28)
float64
0
0
3000 3000


In [43]:
original_accuracy = accuracy_score(y_val, y_pred)
print("Original accuracy:", original_accuracy)

for feature in [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]:
    train_without = df_train.drop(columns=[feature])
    val_without = df_val.drop(columns=[feature])

    dv_temp = DictVectorizer()
    X_train_temp = dv_temp.fit_transform(
        train_without.to_dict(orient='records')
    )
    X_val_temp = dv_temp.transform(
        val_without.to_dict(orient='records')
    )

    model_temp = LogisticRegression(
        solver='liblinear', C=1.0,
        max_iter=1000, random_state=42
    )
    model_temp.fit(X_train_temp, y_train)

    accuracy = accuracy_score(
        y_val, model_temp.predict(X_val_temp)
    )

    print(feature, accuracy,
          "difference:", original_accuracy - accuracy)

Original accuracy: 0.645


ValueError: Only sparse matrices with 32-bit integer indices are accepted. Got int64 indices. Please do report a minimal reproducer on scikit-learn issue tracker so that support for your use-case can be studied by maintainers. See: https://scikit-learn.org/dev/developers/minimal_reproducer.html